# liel × Wikipedia グラフツアー

Stanford SNAP が公開している [**Wikispeedia**](https://snap.stanford.edu/data/wikispeedia.html)
（記事 4,604 / ハイパーリンク 119,882 / ユーザー navigation path 51,318）を使って、
liel で **実データの property graph を構築・問い合わせ・可視化**する一連の流れを体験します。

## カバーする内容

1. データ取得（SNAP から tar.gz を自動 DL、ローカルキャッシュ）
2. pandas で中身を眺める
3. スキーマ設計とプリセット適用
4. liel へのバルク挿入（ファイル永続化）
5. 基本統計・次数分布
6. BFS と最短経路
7. 人間の navigation path との比較（L プリセットのみ）
8. サブグラフ可視化
9. 保存・再オープン

## プリセット

冒頭の `PRESET` を切り替えると処理量が変わります。見積もりセルで実行前に
時間・ディスク・RAM を確認できます。

- **S**: Article + LINKS_TO のみ。チュートリアル標準（数秒）。
- **L**: + category / + NAVIGATED エッジ（数十秒、実データ比較までフル体験）。

> **ライセンス:** Wikispeedia データは CC BY-SA 3.0 で配布されています。詳細は最終セルの参考文献を参照してください。

## 0. セットアップ

- `PRESET` を切り替えると本ノートブック全体の挙動が変わります。
- `_utils.py` は `examples/notebooks/` にある共有モジュールで、DL / 計測 / 見積もりを担当します。

In [ ]:
import sys
from pathlib import Path

# make `_utils` importable regardless of where Jupyter was launched
HERE = Path.cwd() if Path("_utils.py").exists() else Path.cwd() / "examples" / "notebooks"
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import _utils  # noqa: E402
import liel    # noqa: E402
import pandas as pd  # noqa: E402
import networkx as nx  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402

PRESET = "S"   # "S" (fast, ~5s) / "L" (full, ~25s)

print(f"liel version: {liel.__version__}")
print(f"preset     : {PRESET}")
print()
print(_utils.presets_table("01_tour"))

## 1. データ取得 (SNAP Wikispeedia)

初回のみ `tar.gz`（約 10 MB）をダウンロードして展開します。
2 回目以降は `examples/notebooks/data/wikispeedia/` にあるキャッシュを再利用します。

In [ ]:
root = _utils.fetch_wikispeedia()
paths = _utils.wikispeedia_paths(root)

for name, p in paths.items():
    size_kb = p.stat().st_size / 1024
    print(f"  {name:18s} {size_kb:8.1f} KiB   {p.name}")

## 2. 中身を覗く

`articles.tsv` / `links.tsv` / `categories.tsv` は先頭行にヘッダ（コメント）があり、
記事名は URL エンコードされています。ノートブックでは読みやすさのため
`urllib.parse.unquote` でデコードした列も用意します。

In [ ]:
from urllib.parse import unquote

articles_df = pd.read_csv(
    paths["articles"], sep="\t", comment="#",
    header=None, names=["article"], encoding="utf-8",
)
articles_df["title"] = articles_df["article"].map(unquote)
print(f"{len(articles_df):,} articles")
articles_df.head()

In [ ]:
links_df = pd.read_csv(
    paths["links"], sep="\t", comment="#",
    header=None, names=["from", "to"], encoding="utf-8",
)
print(f"{len(links_df):,} directed links")
links_df.head()

In [ ]:
cats_df = pd.read_csv(
    paths["categories"], sep="\t", comment="#",
    header=None, names=["article", "category"], encoding="utf-8",
)
# one article can belong to multiple categories
top_cats = (
    cats_df["category"]
    .map(lambda s: s.split(".")[1] if "." in s else s)
    .value_counts()
    .head(10)
)
print(f"{len(cats_df):,} (article, category) rows / "
      f"{cats_df['article'].nunique():,} articles carry ≥1 category\n")
print("top-level categories:")
top_cats

## 3. スキーマ設計とプリセット適用

このノートブックで使うグラフスキーマ：

| 要素 | ラベル | プロパティ（S / L） |
|---|---|---|
| ノード | `Article` | `title` / `article_key` ／ L なら `top_category` `category_count` も |
| エッジ | `LINKS_TO` | なし |
| エッジ (L のみ) | `NAVIGATED` | なし（頻度は重複エッジ数で表現） |

`NAVIGATED` は `paths_finished.tsv` の navigation path を **連続記事ペアに展開**して
（`<` = Back ボタンは除外）貼るので、**同じ (from, to) が複数回現れ得ます**（マルチグラフ）。

In [ ]:
if PRESET == "S":
    use_categories = False
    use_navigated = False
elif PRESET == "L":
    use_categories = True
    use_navigated = True
else:
    raise ValueError(f"unknown PRESET={PRESET!r}  (expected 'S' or 'L')")

est = _utils.estimate("01_tour", PRESET, "load")
_utils.confirm_run(est, threshold_seconds=60)

## 4. バルク挿入

一時ファイルに `.liel` を作って書き込みます。
`commit()` は WAL を fsync するため **ループ内で呼ばずに段階ごとにまとめる**のが推奨です。

In [ ]:
import shutil
import uuid
from tqdm.auto import tqdm

tmp_root = _utils.DEFAULT_DATA_DIR / "tmp"
tmp_root.mkdir(parents=True, exist_ok=True)
tmpdir = tmp_root / f"liel-wikispeedia-{uuid.uuid4().hex}"
tmpdir.mkdir(parents=True, exist_ok=False)
db_path = tmpdir / "wikispeedia.liel"
db = liel.open(str(db_path))

# article key -> liel node id
name_to_id: dict[str, int] = {}

# Pre-compute primary category per article so that S can skip it cheaply.
primary_cat: dict[str, tuple[str, int]] = {}
if use_categories:
    for art, cat in cats_df.itertuples(index=False):
        top = cat.split(".")[1] if "." in cat else cat
        if art not in primary_cat:
            primary_cat[art] = (top, 1)
        else:
            prev_top, count = primary_cat[art]
            primary_cat[art] = (prev_top, count + 1)

with _utils.Timer("build Article nodes") as t_nodes:
    for key, title in tqdm(
        zip(articles_df["article"], articles_df["title"]),
        total=len(articles_df), desc="articles",
    ):
        props: dict[str, object] = {"title": title, "article_key": key}
        if use_categories:
            pc = primary_cat.get(key)
            if pc is not None:
                props["top_category"] = pc[0]
                props["category_count"] = pc[1]
        name_to_id[key] = db.add_node(["Article"], **props).id
    db.commit()

print(f"nodes so far: {db.node_count():,}")

In [ ]:
# liel の WAL 予約領域は 4 MiB なので、数十万単位のバルク挿入は
# COMMIT_BATCH 件ごとに `commit()` を挟むのが定石です。
# （一発で commit すると TransactionError: WAL overflow が出ます）
COMMIT_BATCH = 20_000

with _utils.Timer("build LINKS_TO edges") as t_links:
    skipped = 0
    pending = 0
    for frm, to in tqdm(
        links_df.itertuples(index=False, name=None),
        total=len(links_df), desc="links",
    ):
        src = name_to_id.get(frm)
        dst = name_to_id.get(to)
        if src is None or dst is None:
            skipped += 1
            continue
        db.add_edge(src, "LINKS_TO", dst)
        pending += 1
        if pending >= COMMIT_BATCH:
            db.commit()
            pending = 0
    if pending:
        db.commit()

print(f"edges so far: {db.edge_count():,}   skipped (dangling): {skipped}")

In [ ]:
t_nav_elapsed = 0.0
nav_paths_df = None
nav_pair_count = 0

if use_navigated:
    # paths_finished.tsv: ip \t timestamp \t duration \t path \t rating
    nav_paths_df = pd.read_csv(
        paths["paths_finished"], sep="\t", comment="#",
        header=None,
        names=["ip", "timestamp", "duration", "path", "rating"],
        encoding="utf-8", keep_default_na=False,
    )
    with _utils.Timer("build NAVIGATED edges") as t_nav:
        pending = 0
        for row in tqdm(
            nav_paths_df.itertuples(index=False),
            total=len(nav_paths_df), desc="navigations",
        ):
            steps = row.path.split(";")
            prev_id: int | None = None
            for step in steps:
                if step == "<":        # back button: keep prev_id so next hop chains from the same article
                    continue
                cur_id = name_to_id.get(step)
                if cur_id is None:
                    prev_id = None
                    continue
                if prev_id is not None:
                    db.add_edge(prev_id, "NAVIGATED", cur_id)
                    nav_pair_count += 1
                    pending += 1
                    if pending >= COMMIT_BATCH:
                        db.commit()
                        pending = 0
                prev_id = cur_id
        if pending:
            db.commit()
    t_nav_elapsed = t_nav.elapsed

    print(f"NAVIGATED edges: {nav_pair_count:,}  "
          f"(from {len(nav_paths_df):,} finished paths)")
else:
    print("PRESET=S: skipping NAVIGATED edges")

In [ ]:
total_elapsed = t_nodes.elapsed + t_links.elapsed + t_nav_elapsed
disk_mb = db_path.stat().st_size / (1024 * 1024)
peak_rss = _utils.current_rss_mb()

print(f"\n=== load summary ({PRESET}) ===")
print(f"  nodes  : {db.node_count():,}")
print(f"  edges  : {db.edge_count():,}")
print(f"  file   : {disk_mb:.2f} MiB  ({db_path})")
print(f"  elapsed: {total_elapsed:.2f} s  "
      f"(nodes {t_nodes.elapsed:.2f} / links {t_links.elapsed:.2f} / nav {t_nav_elapsed:.2f})")
if peak_rss > 0:
    print(f"  RSS    : {peak_rss:.0f} MiB (psutil)")

_utils.save_run({
    "nb": "01_tour", "preset": PRESET, "op": "load",
    "seconds": total_elapsed,
    "disk_mb": disk_mb,
    "peak_mem_mb": peak_rss,
    "n_nodes": db.node_count(),
    "n_edges": db.edge_count(),
})

## 5. 基本統計

`db.info()` と `db.all_edges_as_records()` はどちらも **1 回の PyO3 境界越え**で
まとめて結果を返すので、この粒度の集計は Python レベルのループを挟まず高速です。

In [ ]:
info = db.info()
print(info)

edges_rec = db.all_edges_as_records()
edges_df = pd.DataFrame(edges_rec)
print()
print("edges per label:")
edges_df["label"].value_counts()

## 6. 次数分布

`db.degree_stats()` は `{node_id: (out, in)}` を **Rust 側で計算**して返します。
Python 側で `all_edges()` を舐めるより桁違いに速いので、分布を見るときはこれを使いましょう。

In [ ]:
deg = db.degree_stats()

out_degs = [o for (o, _) in deg.values()]
in_degs = [i for (_, i) in deg.values()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(out_degs, bins=60, color="steelblue")
axes[0].set(title="out-degree distribution",
            xlabel="out-degree", ylabel="# articles", yscale="log")
axes[1].hist(in_degs, bins=60, color="seagreen")
axes[1].set(title="in-degree distribution",
            xlabel="in-degree", ylabel="# articles", yscale="log")
plt.tight_layout()
plt.show()

# top hubs by total degree
total_deg = [(nid, o + i) for nid, (o, i) in deg.items()]
top_hubs = sorted(total_deg, key=lambda t: -t[1])[:10]
hub_df = pd.DataFrame(
    [(db.get_node(nid)["title"], o, i, o + i)
     for nid, _ in top_hubs
     for (o, i) in [deg[nid]]],
    columns=["title", "out", "in", "total"],
)
hub_df

## 7. BFS と最短経路

有名記事をシードに `db.bfs(seed, max_depth=k)` で k-hop 到達範囲を測り、
任意 2 記事ペアの `shortest_path` を LINKS_TO エッジに限定して探します。

In [ ]:
from collections import Counter

candidates = [
    "United_States", "World_War_II", "Europe", "Africa",
    "Mathematics", "Music", "Computer", "Earth",
]
seeds = [c for c in candidates if c in name_to_id]
seed_key = seeds[0] if seeds else next(iter(name_to_id))
seed_id = name_to_id[seed_key]
print(f"seed article: {unquote(seed_key)}")

for k in (1, 2, 3):
    with _utils.Timer(f"BFS depth={k}") as t_bfs:
        reached = db.bfs(seed_id, max_depth=k)
    depth_counts = Counter(d for _, d in reached)
    total = sum(depth_counts.values())
    breakdown = ", ".join(f"d={d}:{depth_counts[d]}" for d in sorted(depth_counts))
    print(f"  reached {total:,} articles in {t_bfs.elapsed*1000:.1f} ms  ({breakdown})")

In [ ]:
def short(a_key: str, b_key: str) -> list[str] | None:
    a, b = name_to_id.get(a_key), name_to_id.get(b_key)
    if a is None or b is None:
        return None
    p = db.shortest_path(a, b, edge_label="LINKS_TO")
    return [n["title"] for n in p] if p else None

pairs = [
    ("Pikachu", "Albert_Einstein"),
    ("Pizza", "Pluto"),
    ("Music", "Mathematics"),
    ("Africa", "United_States"),
]
for a, b in pairs:
    with _utils.Timer(f"{unquote(a)} -> {unquote(b)}") as t:
        path = short(a, b)
    if path is None:
        print("  (不達 or 未登場)")
    else:
        print(f"  hops={len(path)-1}: " + " → ".join(path))

## 8. 実ナビゲーションと最短経路の比較 (L プリセットのみ)

Wikispeedia は「スタート記事 → ゴール記事」を人間がリンクだけで辿るゲームです。
`paths_finished.tsv` には成功したプレイ（ユーザーが辿った path）が並んでいます。
これと liel が返す **LINKS_TO の最短経路**を比べると、
「人間は最短を通らない」ことが素直に見えます。

In [ ]:
if nav_paths_df is None:
    print("PRESET=S ではスキップ。L に切り替えて再実行してください。")
else:
    sample = nav_paths_df.sample(n=min(300, len(nav_paths_df)), random_state=42)
    rows = []
    for row in sample.itertuples(index=False):
        steps = [s for s in row.path.split(";") if s != "<"]
        if len(steps) < 2:
            continue
        a, b = steps[0], steps[-1]
        if a not in name_to_id or b not in name_to_id:
            continue
        sp = db.shortest_path(name_to_id[a], name_to_id[b], edge_label="LINKS_TO")
        if sp is None:
            continue
        rows.append({
            "human_hops": len(steps) - 1,
            "shortest_hops": len(sp) - 1,
            "overhead": (len(steps) - 1) - (len(sp) - 1),
        })

    cmp_df = pd.DataFrame(rows)
    print(f"比較サンプル数: {len(cmp_df):,}")
    print(cmp_df.describe().round(2))

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(cmp_df["shortest_hops"], cmp_df["human_hops"],
               alpha=0.35, s=24, color="steelblue")
    lim = max(cmp_df["human_hops"].max(), cmp_df["shortest_hops"].max()) + 1
    ax.plot([0, lim], [0, lim], "k--", alpha=0.4, label="optimal (y=x)")
    ax.set(xlabel="liel shortest path (hops)",
           ylabel="human navigation (hops)",
           title="人間は最短経路を通らない")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 9. サブグラフ可視化

`degree_stats()` で得た上位 30 hub と、`edges_between(ids)` で抽出した
そのサブグラフだけを networkx に渡して描画します。

In [ ]:
top_ids = [nid for nid, _ in sorted(
    ((nid, o + i) for nid, (o, i) in deg.items()),
    key=lambda t: -t[1],
)[:30]]

sub_edges = db.edges_between(top_ids)
G = nx.DiGraph()
for nid in top_ids:
    node = db.get_node(nid)
    G.add_node(nid, title=node["title"])
for e in sub_edges:
    if e["label"] != "LINKS_TO":
        continue
    G.add_edge(e["from_node"], e["to_node"])

pos = nx.spring_layout(G, seed=42, k=0.8)
sizes = [400 + 15 * (deg[n][0] + deg[n][1]) for n in G.nodes()]
labels = {n: G.nodes[n]["title"][:22] for n in G.nodes()}

fig, ax = plt.subplots(figsize=(12, 8))
nx.draw_networkx_nodes(G, pos, node_color="lightsteelblue",
                       node_size=sizes, edgecolors="#556", ax=ax)
nx.draw_networkx_labels(G, pos, labels=labels, font_size=7, ax=ax)
nx.draw_networkx_edges(G, pos, arrows=True, edge_color="#aaa",
                       alpha=0.45, arrowsize=10, ax=ax,
                       connectionstyle="arc3,rad=0.08")
ax.set_title(f"Top 30 hub articles — LINKS_TO subgraph ({PRESET})")
ax.axis("off")
plt.tight_layout()
plt.show()

## 10. 保存・再オープン

`.liel` は単一ファイルです。`close()` してから同じパスを `open()` すれば
ノード ID も含めて完全に復元されます。

In [ ]:
db.close()
size_mb = db_path.stat().st_size / (1024 * 1024)
print(f"closed.  file: {size_mb:.2f} MiB  path: {db_path}")

db2 = liel.open(str(db_path))
print(f"reopened: {db2.node_count():,} nodes, {db2.edge_count():,} edges")

# IDs are stable across reopen, so we can reuse our name_to_id map.
print(f"\nBFS depth=1 from '{unquote(seed_key)}' (after reopen):")
for n, d in db2.bfs(name_to_id[seed_key], max_depth=1)[:5]:
    print(f"  depth={d}  {n['title']}")

db2.close()

## 11. クリーンアップ

一時 `.liel` ファイルを削除します。
`examples/notebooks/data/wikispeedia/` のダウンロードはキャッシュとして残ります
（.gitignore 対象なので Git には入りません）。

In [ ]:
shutil.rmtree(tmpdir, ignore_errors=True)
print("removed tmpdir:", tmpdir)

## 参考

- **Wikispeedia dataset** (SNAP):
  https://snap.stanford.edu/data/wikispeedia.html
- West, R., & Leskovec, J. (2012). *Human Wayfinding in Information Networks.* WWW.
- West, R., Pineau, J., & Precup, D. (2009). *Wikispeedia: An Online Game for Inferring
  Semantic Distances between Concepts.* IJCAI.
- データライセンス: CC BY-SA 3.0

このノートで使った liel API：
`liel.open`, `add_node`, `add_edge`, `commit`, `close`,
`node_count`, `edge_count`, `info`,
`all_edges_as_records`, `degree_stats`, `edges_between`,
`get_node`, `bfs`, `shortest_path`.